# Project Hilbert-Link - Data Preparation (QM9)

## Dataset: QM9
~134k small organic molecules with C, H, O, N, and F atoms.
Suitable for molecular graph generation models.

## Applied Filters
1. **RDKit validity** - drops SMILES that cannot be parsed
2. **Sanitization** - drops molecules that fail chemical sanitization
3. **Heavy atoms <= NUM_ATOMS** - matches the MolGAN featurizer
4. **Allowed atom types** - only {C, N, O, F} with implicit hydrogens
5. **Connected molecule** - drops disconnected SMILES
6. **Molecular weight <= 200 Da** - focuses on small generated molecules
7. **Duplicate removal** - keeps unique canonical SMILES
8. **Valid GraphMatrix** - drops failed featurizations


In [ ]:
import sys
print(f'Python version: {sys.version}')
print(f'Executable path: {sys.executable}')

In [ ]:
import config

In [ ]:
\
BASE_DIR = 'data'

import os
os.makedirs(BASE_DIR, exist_ok=True)
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
!{sys.executable} -m pip install --pre deepchem --quiet
!{sys.executable} -m pip install rdkit fcd_torch pennylane --quiet
!{sys.executable} -m pip install tensorflow --quiet
!{sys.executable} -m pip install tqdm torch matplotlib --quiet

In [ ]:
import warnings
from rdkit import RDLogger

\
warnings.filterwarnings('ignore', category=DeprecationWarning)

\
logger = RDLogger.logger()
logger.setLevel(RDLogger.CRITICAL)

import logging

\
logger = logging.getLogger('deepchem')
logger.setLevel(logging.ERROR)

In [ ]:
\
import random, numpy as np, torch
SEED = config.SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Seeds synchronized.')

In [ ]:
\
import pickle, pandas as pd
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from deepchem.feat.molecule_featurizers.molgan_featurizer import GraphMatrix

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'data'))
import config
from config import *

In [ ]:
\
\
\
print('Baixando/carregando QM9 (pode demorar na primeira vez)...')
tasks, datasets, transformers = dc.molnet.load_qm9(splitter='random')

\
\
all_smiles = list(datasets[0].ids) + list(datasets[1].ids) + list(datasets[2].ids)
df = pd.DataFrame({'smiles': all_smiles})
print(f'Raw molecules loaded: {len(df)}')

In [ ]:
\

\
ALLOWED_ATOMS = {'C', 'N', 'O', 'F'}
MAX_MW        = 200.0                                                     
MAX_ATOMS     = config.NUM_ATOMS                               

def passes_filters(smi: str) -> bool:
    """
    Return True when a molecule passes all quality filters
    para treinamento de modelos generativos.
    """
    \
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return False

    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return False

    if '.' in smi:
        return False

    if mol.GetNumAtoms() >= MAX_ATOMS:
        return False

    atom_symbols = {atom.GetSymbol() for atom in mol.GetAtoms()}
    if not atom_symbols.issubset(ALLOWED_ATOMS):
        return False

    if Descriptors.MolWt(mol) > MAX_MW:
        return False

    return True

print('Aplicando filtros...')
n0 = len(df)

\
df['canon_smiles'] = df['smiles'].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s))
    if Chem.MolFromSmiles(s) else None
)
df.dropna(subset=['canon_smiles'], inplace=True)
df.drop_duplicates(subset=['canon_smiles'], inplace=True)
n1 = len(df)
print(f'  After deduplication: {n1:,} (-{n0 - n1:,})')

\
mask = df['canon_smiles'].apply(passes_filters)
df_filtered = df[mask].copy()
n2 = len(df_filtered)
print(f'  After quality filters: {n2:,} (-{n1 - n2:,})')

filtered_smiles = df_filtered['canon_smiles'].tolist()
print(f'
Total after all filters: {len(filtered_smiles):,}')

In [ ]:
\
feat = dc.feat.MolGanFeaturizer(
    max_atom_count=MAX_ATOMS,
    atom_labels=config.ATOM_LABELS
)

features = feat.featurize(filtered_smiles)

\
indices      = [i for i, d in enumerate(features) if type(d) is GraphMatrix]
features     = [features[i] for i in indices]
train_smiles = [filtered_smiles[i] for i in indices]

print(f'Valid GraphMatrix featurizations: {len(features):,}')
print(f'train set final:                     {len(train_smiles):,}')

In [ ]:
\
import matplotlib.pyplot as plt

mols        = [Chem.MolFromSmiles(s) for s in train_smiles]
mws         = [Descriptors.MolWt(m)  for m in mols if m]
atom_counts = [m.GetNumAtoms()        for m in mols if m]

\
from collections import Counter
atom_type_counter = Counter(
    atom.GetSymbol()
    for m in mols if m
    for atom in m.GetAtoms()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(mws, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Molecular Weight Distribution (train set)')
axes[0].set_xlabel('MW (Da)')
axes[0].set_ylabel('Count')

axes[1].hist(atom_counts, bins=range(1, MAX_ATOMS + 2), color='coral', edgecolor='white')
axes[1].set_title('Atom Count Distribution (train set)')
axes[1].set_xlabel('Num Heavy Atoms')
axes[1].set_ylabel('Count')

labels  = list(atom_type_counter.keys())
values  = list(atom_type_counter.values())
colors  = ['#4CAF50', '#2196F3', '#FF5722', '#9C27B0']
axes[2].bar(labels, values, color=colors[:len(labels)], edgecolor='white')
axes[2].set_title('Atom Type Frequency (train set)')
axes[2].set_xlabel('Atom Type')
axes[2].set_ylabel('Total Count')

plt.suptitle(f'QM9 - {len(train_smiles):,} molecules after filtering', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'data_exploration.png'), dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
\
print('=' * 50)
print('FILTERING PIPELINE SUMMARY (QM9)')
print('=' * 50)
print(f'  Raw molecules loaded : {n0:>8,}')
print(f'  After deduplication  : {n1:>8,}')
print(f'  After quality filters: {n2:>8,}')
print(f'  After featurization  : {len(train_smiles):>8,}')
print('=' * 50)
print(f'  MAX_ATOMS     : {MAX_ATOMS}')
print(f'  MAX_MW        : {MAX_MW} Da')
print(f'  ALLOWED_ATOMS : {sorted(ALLOWED_ATOMS)}')
print('=' * 50)

In [ ]:
len(train_smiles)

In [ ]:
\
import numpy as np

data_payload = {
    'train_smiles':  train_smiles,
    'adj_matrices': [x.adjacency_matrix.tolist() for x in features],
    'node_features': [x.node_features.tolist()    for x in features],
}

save_path = os.path.join(BASE_DIR, 'prepared_data.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(data_payload, f)

print(f'Saved {len(train_smiles):,} molecules → {save_path}')